# 05 — HayFlow-Hines prototype

Questo notebook implementa il primo core HayFlow realmente morphology-aware. Non riutilizza il forward di B3: usa `U_realized`, stato ricorrente locale e globale, un singolo solve Hines differenziabile, teste evento separate e una jump map locale. B3 resta soltanto un riferimento numerico.

Il gate principale è l'**overfit canary**. Se HayFlow-Hines non riesce a memorizzare il piccolo insieme bilanciato, il training completo viene fermato e il problema viene classificato prima di spendere ore di GPU.

## 1. Checkout coerente e dipendenze
Notebook, librerie e configurazione devono provenire dalla stessa revisione Git.

In [ ]:
import os, subprocess, sys
from pathlib import Path

WORKSPACE = Path('/kaggle/working/hayflow_workspace')
ELM_REPO = WORKSPACE / 'elmneuron'
ELM_REF = os.environ.get('HAYFLOW_ELM_REF', 'main')
WORKSPACE.mkdir(parents=True, exist_ok=True)
if not ELM_REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/Zagred47/giada.git', str(ELM_REPO)], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'fetch', 'origin', ELM_REF], check=True)
subprocess.run(['git', '-C', str(ELM_REPO), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
REVISION = subprocess.check_output(['git', '-C', str(ELM_REPO), 'rev-parse', 'HEAD'], text=True).strip()
sys.path.insert(0, str(ELM_REPO))
print('Revisione coerente caricata:', REVISION)

In [ ]:
import h5py, json, numpy, pandas, pyarrow, torch, yaml
print({
    'h5py': h5py.__version__, 'numpy': numpy.__version__,
    'pandas': pandas.__version__, 'pyarrow': pyarrow.__version__,
    'torch': torch.__version__, 'cuda': torch.cuda.is_available(),
    'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
})
assert torch.cuda.is_available(), 'Il profilo decisionale 05 richiede una GPU Kaggle.'

## 2. Dataset composito e riferimento B3
Aggiungere agli input Kaggle: dataset base targeted v1.1, top-up v3 e il piccolo ZIP `hayflow_release_identifiability_flowmap_v1_1.zip`. Non vengono generati nuovi dati.

In [ ]:
import zipfile, shutil

INPUT_ROOT = Path('/kaggle/input')
topup_override = os.environ.get('HAYFLOW_TOPUP_V3')
topup_candidates = [Path(topup_override).expanduser()] if topup_override else []
topup_candidates.extend(INPUT_ROOT.rglob('hayflow_bap_validation_support_topup_v3.zip'))
topup_candidates.extend(path.parent for path in INPUT_ROOT.rglob('composite_dataset_manifest.json'))
TOPUP_SOURCE = next((path.resolve() for path in topup_candidates if path.exists()), None)
assert TOPUP_SOURCE is not None, 'Top-up v3 non trovato negli input Kaggle.'

def extract_zip_safely(source, destination):
    destination = Path(destination)
    marker = destination / '.source_size'
    stamp = str(Path(source).stat().st_size)
    if marker.is_file() and marker.read_text().strip() == stamp:
        return destination
    if destination.exists(): shutil.rmtree(destination)
    destination.mkdir(parents=True)
    root = destination.resolve()
    with zipfile.ZipFile(source) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            assert target == root or root in target.parents, member.filename
        archive.extractall(destination)
    marker.write_text(stamp)
    return destination

TOPUP_ROOT = extract_zip_safely(TOPUP_SOURCE, '/kaggle/working/hayflow05_topup') if TOPUP_SOURCE.is_file() else TOPUP_SOURCE
manifest_candidates = list(Path(TOPUP_ROOT).rglob('composite_dataset_manifest.json'))
assert len(manifest_candidates) == 1, manifest_candidates
COMPOSITE_MANIFEST = manifest_candidates[0]

base_override = os.environ.get('HAYFLOW_BASE_DATASET')
base_candidates = [Path(base_override).expanduser()] if base_override else []
base_candidates.extend(path.parent for path in INPUT_ROOT.rglob('transition_dataset.h5') if 'targeted' in str(path).lower() and 'topup' not in str(path).lower())
base_candidates.extend(path for path in INPUT_ROOT.rglob('archive.zip') if 'hayflow-targeted-transition-dataset' in str(path).lower())
BASE_SOURCE = next((path.resolve() for path in base_candidates if path.exists()), None)
assert BASE_SOURCE is not None, 'Dataset base targeted v1.1 non trovato.'

b3_override = os.environ.get('HAYFLOW_B3_RESULT')
b3_candidates = [Path(b3_override).expanduser()] if b3_override else []
b3_candidates.extend(INPUT_ROOT.rglob('hayflow_release_identifiability_flowmap_v1_1.zip'))
b3_candidates.extend(path.parent for path in INPUT_ROOT.rglob('final_report.json') if 'release' in str(path).lower())
B3_SOURCE = next((path.resolve() for path in b3_candidates if path.exists()), None)
assert B3_SOURCE is not None, 'Risultato 04/B3 non trovato negli input Kaggle.'
if B3_SOURCE.is_file():
    B3_ROOT = extract_zip_safely(B3_SOURCE, '/kaggle/working/hayflow05_b3')
else:
    B3_ROOT = B3_SOURCE
b3_reports = list(Path(B3_ROOT).rglob('final_report.json'))
assert len(b3_reports) == 1, b3_reports
B3_REPORT = json.loads(b3_reports[0].read_text())
print({'manifest': str(COMPOSITE_MANIFEST), 'base': str(BASE_SOURCE), 'b3': str(b3_reports[0])})

## 3. Verifica crittografica e preflight
Gli shard restano fisicamente separati. Il tracker mostra avanzamento ed ETA dell'hash del base da circa 6 GiB.

In [ ]:
import time
from src.hayflow_data import prepare_composite_flowmap_bundle

hash_started, hash_last = {}, {}
def hash_progress(name, done, total):
    now = time.monotonic(); hash_started.setdefault(name, now)
    percent = int(100 * done / total)
    if percent >= hash_last.get(name, -5) + 5 or done == total:
        elapsed = now - hash_started[name]; rate = done / max(elapsed, 1e-9)
        eta = (total - done) / max(rate, 1e-9)
        print(f'[HayFlow 05][SHA-256 {name}] {percent}% ETA {eta/60:.1f} min', flush=True)
        hash_last[name] = percent

bundle = prepare_composite_flowmap_bundle(
    COMPOSITE_MANIFEST, base_source=BASE_SOURCE,
    cache_dir=Path('/kaggle/working/hayflow05_cache'),
    verify_hashes=True, progress=hash_progress,
)
print({'fingerprint': bundle.fingerprint, 'transitions': bundle.transition_count, 'physical_merge': False})

In [ ]:
from src.hayflow_model import HayFlowHinesExperiment, HinesPrototypeExperimentConfig

raw = yaml.safe_load((ELM_REPO / 'configs/hayflow/hayflow_hines_prototype.yml').read_text())
experiment_config = dict(raw['experiment'])
experiment_config['profile'] = os.environ.get('HAYFLOW_05_PROFILE', experiment_config['profile'])
config = HinesPrototypeExperimentConfig.from_mapping(experiment_config)
OUTPUT_DIR = Path('/kaggle/working/artifacts/hayflow_hines_prototype')
session = HayFlowHinesExperiment(bundle, OUTPUT_DIR, config, b3_report=B3_REPORT)
prepare_report = session.prepare()
display(prepare_report)
display({'training_branch_pair': prepare_report['training_branch_pair'],
         'training_contract_blockers': prepare_report['training_contract_blockers']})
assert prepare_report['loader']['episode_count'] == 369
assert prepare_report['loader']['transition_count'] == 29880
assert prepare_report['loader']['topup_validation_only']

## 4. Test indipendenti del layer Hines
Il notebook non può procedere se eliminazione su albero, solve denso, gradcheck o sicurezza della morfologia canonica non coincidono.

In [ ]:
hines_report = session.run_hines_layer_tests()
display(hines_report)
assert hines_report['valid']

## 5. Overfit canary obbligatorio
Questa cella allena HayFlow-Hines H2 e la ConvGRU convenzionale sul piccolo insieme bilanciato. Stampa epoca, loss, RMSE, F1 minimo ed ETA.

In [ ]:
canary_report = session.run_canary()
display({
    'scenario': canary_report['scenario'],
    'proceed_to_full_training': canary_report['proceed_to_full_training'],
    'event_support': canary_report['event_support'],
    'models': {name: {key: value for key, value in report.items() if key != 'history'} for name, report in canary_report['models'].items()},
})

In [ ]:
if not canary_report['proceed_to_full_training']:
    stopped_report = session.finalize(canary_report)
    display(stopped_report)
    raise RuntimeError(
        'Canary HayFlow-Hines non superato: training completo fermato. '
        f"Scenario: {canary_report['scenario']}. Scaricare gli artefatti diagnostici."
    )
print('Canary superato: il curriculum completo è autorizzato.')

## 6. Curriculum completo e confronto
Questa è la cella lunga. Allena H0/H1/H2 e ConvGRU sul primo seed; H2 e ConvGRU sugli altri seed. Esegue one-step, rollout stratificati 2/4/8/16/32 ms, eventi, branching e recovery. I checkpoint `one_step`, `event` e rollout sono distinti.

In [ ]:
final_report = session.run_full(canary_report)
display({
    'valid': final_report['valid'],
    'decision_grade': final_report['decision_grade'],
    'scenario': final_report['scenario'],
    'hayflow_hines': final_report['hayflow_hines'],
    'convgru': final_report['convgru'],
    'b3_reference': final_report['b3_reference'],
    'training_contract': final_report['training_contract'],
})

## 7. Controllo documentale degli output

In [ ]:
required = [
    'hines_layer_tests.json', 'canary_overfit_report.json',
    'model_configurations.json', 'composite_loader_report.json',
    'normalization_schema.json', 'one_step_metrics.parquet',
    'rollout_metrics.parquet', 'event_metrics.parquet',
    'branching_metrics.parquet', 'recovery_metrics.parquet',
    'regional_drift.parquet', 'peak_attenuation.parquet',
    'model_comparison.parquet', 'out_of_domain_metrics.parquet',
    'checkpoint_registry.json', 'final_report.json',
]
missing = [name for name in required if not (OUTPUT_DIR / name).is_file()]
assert not missing, missing
print('Output completi:', len(required), '| checkpoint:', len(list((OUTPUT_DIR / 'checkpoints').rglob('*.pt'))))

## 8. Download nel browser
Metodo Kaggle già validato nel progetto: ZIP in `/kaggle/working`, Base64, JavaScript, Blob e click. Di default esclude i checkpoint; impostare `HAYFLOW_DOWNLOAD_CHECKPOINTS=1` per includerli.

In [ ]:
from shutil import copytree, make_archive, rmtree
import base64
from IPython.display import Javascript, display

include_checkpoints = os.environ.get('HAYFLOW_DOWNLOAD_CHECKPOINTS', '0') == '1'
archive_source = OUTPUT_DIR
staging = Path('/kaggle/working/hayflow05_download')
if not include_checkpoints:
    if staging.exists(): rmtree(staging)
    copytree(OUTPUT_DIR, staging, ignore=lambda path, names: {'checkpoints'} if 'checkpoints' in names else set())
    archive_source = staging
zip_base = Path('/kaggle/working/hayflow_hines_prototype')
zip_path = Path(make_archive(str(zip_base), 'zip', root_dir=archive_source.parent, base_dir=archive_source.name))
encoded = base64.b64encode(zip_path.read_bytes()).decode('ascii')
filename = zip_path.name
display(Javascript(f'''
const binary = atob('{encoded}');
const bytes = new Uint8Array(binary.length);
for (let i = 0; i < binary.length; i++) bytes[i] = binary.charCodeAt(i);
const blob = new Blob([bytes], {{type: 'application/zip'}});
const url = URL.createObjectURL(blob);
const anchor = document.createElement('a');
anchor.href = url; anchor.download = '{filename}';
document.body.appendChild(anchor); anchor.click(); anchor.remove();
setTimeout(() => URL.revokeObjectURL(url), 60000);
'''))
print('Download avviato:', zip_path, f'({zip_path.stat().st_size / 2**20:.1f} MiB)')